# 01 — Joint encoder–decoder training

Train four independent end-to-end reconstruction systems on one fixed subject-disjoint split. Every run starts from the same train-only SimCLR encoder, but the encoder, fusion, 3D lift, and decoder all receive reconstruction gradients from epoch 1.

This notebook never constructs a test-set `Dataset`. Set `RUN_TRAINING = True` after the SimCLR checkpoint from notebook 00 has been created.

In [1]:
import contextlib
import gc
import json
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as activation_checkpoint
import timm
from torch.utils.data import DataLoader, Dataset

SEED = 42
RUN_TRAINING = True
RESUME_TRAINING = True
ARMS_TO_RUN = [
    "plain_unet_style",
    "plain_prelu_style",
    "residual_relu_style",
    "residual_vnet_style",
]
ARM_FACTORS = {
    "plain_unet_style": {"activation": "relu", "residual": False, "label": "U (ReLU, plain)"},
    "plain_prelu_style": {"activation": "prelu", "residual": False, "label": "PReLU + plain"},
    "residual_relu_style": {"activation": "relu", "residual": True, "label": "ReLU + residual"},
    "residual_vnet_style": {"activation": "prelu", "residual": True, "label": "V (PReLU, residual)"},
}
BONES = ["femur", "tibia", "patella", "fibula"]
FEATURE_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attention", "attention"]
IMAGE_SIZE = 256
LOGIT_SIZE = 128
BATCH_SIZE = 1
NUM_WORKERS = 4
MAX_EPOCHS = 50
PATIENCE = 10
WARMUP_EPOCHS = 2
ENCODER_LR = 3e-5
NEW_MODULE_LR = 3e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_ACTIVATION_CHECKPOINTING = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: this notebook is intended for the HPC CUDA environment.")
print({"device": str(DEVICE), "run_training": RUN_TRAINING, "arms": ARMS_TO_RUN})

{'device': 'cuda', 'run_training': True, 'arms': ['plain_unet_style', 'plain_prelu_style', 'residual_relu_style', 'residual_vnet_style']}


In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "reports" / "manifests" / "quantitative_manifest_v1.csv").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


ROOT = find_project_root(Path.cwd())
MANIFEST_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.csv"
MODEL_ROOT = ROOT / "models" / "joint_simclr_test"
SIMCLR_PATH = MODEL_ROOT / "simclr" / "simclr_encoder.pth"


def load_fixed_split():
    rows = pd.read_csv(MANIFEST_PATH, dtype={"test_fold": "Int64"})
    rows = rows.loc[rows.status.eq("ready")].copy()
    if len(rows) != 71:
        raise RuntimeError(f"Expected 71 ready knees, found {len(rows)}.")
    rows["test_fold"] = rows.test_fold.astype(int)
    rows["split"] = "train"
    rows.loc[rows.test_fold.eq(1), "split"] = "validation"
    rows.loc[rows.test_fold.eq(0), "split"] = "test"
    expected = {"train": 42, "validation": 14, "test": 15}
    observed = rows.groupby("split").size().to_dict()
    if observed != expected:
        raise RuntimeError(f"Fixed split mismatch: expected {expected}, found {observed}.")
    subjects = {name: set(group.subject_id) for name, group in rows.groupby("split")}
    if subjects["train"] & subjects["validation"] or subjects["train"] & subjects["test"] or subjects["validation"] & subjects["test"]:
        raise RuntimeError("Subject overlap detected between train, validation, and test roles.")
    return {name: rows.loc[rows.split.eq(name)].copy() for name in expected}


splits = load_fixed_split()
display(pd.DataFrame([
    {"split": name, "knees": len(frame), "subjects": frame.subject_id.nunique(), "fractured": int(frame.dataset.eq("Ruikar").sum())}
    for name, frame in splits.items()
]))
seed_everything()

,split,knees,subjects,fractured
0,train,42,26,8
1,validation,14,8,2
2,test,15,9,3


In [3]:
AUGMENTATION = {
    "gamma": (0.90, 1.10),
    "brightness": (-0.05, 0.05),
    "noise_sigma": (0.0, 0.02),
}


def read_drr(path: Path) -> np.ndarray:
    array = np.load(path).astype(np.float32)
    if array.shape != (IMAGE_SIZE, IMAGE_SIZE) or not np.isfinite(array).all():
        raise ValueError(f"Invalid DRR: {path}")
    return np.clip(array, 0.0, 1.0)


def load_target(row) -> torch.Tensor:
    arrays = []
    target_dir = ROOT / row.target_path
    for bone in BONES:
        image = nib.load(str(target_dir / f"{row.sample_id}_{bone}.nii.gz"))
        if image.shape != (IMAGE_SIZE, IMAGE_SIZE, IMAGE_SIZE):
            raise ValueError(f"Target shape mismatch for {row.sample_id}/{bone}: {image.shape}")
        if tuple(nib.aff2axcodes(image.affine)) != ("L", "P", "S"):
            raise ValueError(f"Target orientation mismatch for {row.sample_id}/{bone}")
        array = np.asarray(image.dataobj, dtype=np.float32)
        if not np.isfinite(array).all() or array.sum() <= 0 or not set(np.unique(array).tolist()).issubset({0.0, 1.0}):
            raise ValueError(f"Target must be finite, binary, and non-empty: {row.sample_id}/{bone}")
        arrays.append(array)
    return torch.from_numpy(np.stack(arrays).astype(np.float32))


def augment_drr(array: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    gamma = rng.uniform(*AUGMENTATION["gamma"])
    brightness = rng.uniform(*AUGMENTATION["brightness"])
    sigma = rng.uniform(*AUGMENTATION["noise_sigma"])
    output = np.power(np.clip(array, 0, 1), gamma, dtype=np.float32) + np.float32(brightness)
    if sigma > 0:
        output += rng.normal(0, sigma, output.shape).astype(np.float32)
    return np.clip(output, 0, 1).astype(np.float32)


class ReconstructionDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, training: bool):
        self.rows = rows.reset_index(drop=True)
        self.training = training
        self.epoch = 0

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        ap = read_drr(ROOT / row.ap_drr_path)
        lat = read_drr(ROOT / row.lat_drr_path)
        if self.training:
            base_seed = SEED + self.epoch * 100_000 + index * 2
            ap = augment_drr(ap, base_seed)
            lat = augment_drr(lat, base_seed + 1)
        return {
            "ap": torch.from_numpy(ap).unsqueeze(0),
            "lat": torch.from_numpy(lat).unsqueeze(0),
            "target": load_target(row),
            "sample_id": str(row.sample_id),
            "subject_id": str(row.subject_id),
            "dataset": str(row.dataset),
        }


def training_pos_weight(rows: pd.DataFrame) -> torch.Tensor:
    positives = torch.zeros(len(BONES), dtype=torch.float64)
    total_voxels = 0
    for row in rows.itertuples(index=False):
        target = load_target(row)
        positives += target.sum(dim=(1, 2, 3), dtype=torch.float64)
        total_voxels += target[0].numel()
    negatives = total_voxels - positives
    if bool((positives <= 0).any()) or bool((negatives <= 0).any()):
        raise RuntimeError("Invalid foreground counts in the training split.")
    return (negatives / positives).float().view(1, len(BONES), 1, 1, 1)


def dice_bce_loss(logits, target, pos_weight):
    logits = logits.float()
    target = target.float()
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight)
    probability = torch.sigmoid(logits).flatten(2)
    flattened = target.flatten(2)
    intersection = (probability * flattened).sum(-1)
    dice = (2 * intersection + 1.0) / (probability.sum(-1) + flattened.sum(-1) + 1.0)
    return 0.5 * bce + 0.5 * (1.0 - dice.mean())


@torch.no_grad()
def hard_dice(logits, target):
    prediction = (torch.sigmoid(logits.float()) > 0.5).float().flatten(2)
    target = (target > 0.5).float().flatten(2)
    intersection = (prediction * target).sum(-1)
    return (2 * intersection + 1e-6) / (prediction.sum(-1) + target.sum(-1) + 1e-6)

In [4]:
def make_activation(kind: str, channels: int):
    if kind == "relu":
        return nn.ReLU(inplace=True)
    if kind == "prelu":
        return nn.PReLU(channels)
    raise ValueError(f"Unknown activation: {kind}")


class CrossAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.query = nn.Linear(channels, channels)
        self.key = nn.Linear(channels, channels)
        self.value = nn.Linear(channels, channels)
        self.scale = channels ** -0.5

    def forward(self, query_map, context_map):
        batch, channels, height, width = query_map.shape
        query = query_map.flatten(2).transpose(1, 2)
        context = context_map.flatten(2).transpose(1, 2)
        weights = torch.softmax(self.query(query) @ self.key(context).transpose(-2, -1) * self.scale, dim=-1)
        output = weights @ self.value(context) + query
        return output.transpose(1, 2).reshape(batch, channels, height, width)


class LocalFusion(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.mix = nn.Conv2d(2 * channels, channels, 3, padding=1)

    def forward(self, query_map, context_map):
        return self.mix(torch.cat([query_map, context_map], dim=1)) + query_map


class DecoderBlock(nn.Module):
    def __init__(self, input_channels: int, output_channels: int, activation: str, residual: bool):
        super().__init__()
        self.residual = residual
        self.projection = None
        if residual:
            self.projection = nn.Conv3d(input_channels, output_channels, 1) if input_channels != output_channels else nn.Identity()
        self.conv1 = nn.Conv3d(input_channels, output_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, output_channels)
        self.act1 = make_activation(activation, output_channels)
        self.conv2 = nn.Conv3d(output_channels, output_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, output_channels)
        self.act2 = make_activation(activation, output_channels)

    def forward(self, value):
        residual = self.projection(value) if self.residual else None
        value = self.act1(self.norm1(self.conv1(value)))
        value = self.norm2(self.conv2(value))
        if residual is not None:
            value = value + residual
        return self.act2(value)


class MatchedDecoder3D(nn.Module):
    def __init__(self, arm: str):
        super().__init__()
        factors = ARM_FACTORS[arm]
        activation, residual = factors["activation"], factors["residual"]
        c0, c1, c2, c3 = FEATURE_CHANNELS
        self.up3 = nn.ConvTranspose3d(c3, c2, 2, 2)
        self.dec3 = DecoderBlock(c2 + c2, c2, activation, residual)
        self.up2 = nn.ConvTranspose3d(c2, c1, 2, 2)
        self.dec2 = DecoderBlock(c1 + c1, c1, activation, residual)
        self.up1 = nn.ConvTranspose3d(c1, c0, 2, 2)
        self.dec1 = DecoderBlock(c0 + c0, c0, activation, residual)
        self.output = nn.Conv3d(c0, len(BONES), 1)

    def run_block(self, block, value):
        if USE_ACTIVATION_CHECKPOINTING and self.training and value.requires_grad:
            return activation_checkpoint.checkpoint(block, value, use_reentrant=False)
        return block(value)

    def forward(self, levels):
        level0, level1, level2, level3 = levels
        value = self.run_block(self.dec3, torch.cat([self.up3(level3), level2], dim=1))
        value = self.run_block(self.dec2, torch.cat([self.up2(value), level1], dim=1))
        value = self.run_block(self.dec1, torch.cat([self.up1(value), level0], dim=1))
        value = F.interpolate(value, size=(LOGIT_SIZE,) * 3, mode="trilinear", align_corners=False)
        logits = self.output(value)
        return F.interpolate(logits, size=(IMAGE_SIZE,) * 3, mode="trilinear", align_corners=False)


class JointReconstructionModel(nn.Module):
    def __init__(self, arm: str, simclr_checkpoint):
        super().__init__()
        if arm not in ARM_FACTORS:
            raise ValueError(f"Unknown decoder arm: {arm}")
        model_name = simclr_checkpoint["model_name"]
        self.encoder = timm.create_model(model_name, pretrained=False, num_classes=0)
        self.encoder.load_state_dict(simclr_checkpoint["encoder_state"], strict=True)
        self.normalization = simclr_checkpoint["normalization"]
        source_channels = [96, 192, 384, 768]
        self.fusion = nn.ModuleList([
            CrossAttention(channels) if kind == "attention" else LocalFusion(channels)
            for channels, kind in zip(source_channels, FUSION_TYPES)
        ])
        self.project_2d = nn.ModuleList([
            nn.Conv2d(source, target, 1) for source, target in zip(source_channels, FEATURE_CHANNELS)
        ])
        self.fuse_3d = nn.ModuleList([
            nn.Conv3d(2 * channels, channels, 3, padding=1) for channels in FEATURE_CHANNELS
        ])
        self.decoder = MatchedDecoder3D(arm)

    def normalize(self, raw):
        image = raw.repeat(1, 3, 1, 1)
        mean = torch.tensor(self.normalization["mean"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1)
        std = torch.tensor(self.normalization["std"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1)
        return (image - mean) / std

    def encoder_features(self, image):
        _, levels = self.encoder.forward_intermediates(image, indices=(0, 1, 2, 3))
        return tuple(levels)

    def encode(self, raw):
        image = self.normalize(raw)
        if USE_ACTIVATION_CHECKPOINTING and self.training:
            return activation_checkpoint.checkpoint(self.encoder_features, image, use_reentrant=False)
        return self.encoder_features(image)

    @staticmethod
    def lift(ap_feature, lat_feature, projection, fusion_3d):
        ap = projection(ap_feature)
        lat = projection(lat_feature).flip(3)
        batch, channels, size, _ = ap.shape
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(batch, channels, size, size, size)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(batch, channels, size, size, size)
        combined = torch.cat([ap_cube, lat_cube], dim=1)
        if USE_ACTIVATION_CHECKPOINTING and combined.requires_grad:
            return activation_checkpoint.checkpoint(fusion_3d, combined, use_reentrant=False)
        return fusion_3d(combined)

    def forward(self, ap_raw, lat_raw):
        ap_levels = self.encode(ap_raw)
        lat_levels = self.encode(lat_raw)
        output = []
        for ap, lat, fusion, projection, fusion_3d in zip(ap_levels, lat_levels, self.fusion, self.project_2d, self.fuse_3d):
            fused_ap = fusion(ap, lat)
            fused_lat = fusion(lat, ap)
            output.append(self.lift(fused_ap, fused_lat, projection, fusion_3d))
        logits = self.decoder(output)
        if logits.shape[1:] != (len(BONES), IMAGE_SIZE, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f"Unexpected model output shape: {tuple(logits.shape)}")
        return logits

In [5]:
def amp_context():
    if not (USE_AMP and DEVICE.type == "cuda"):
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast("cuda", dtype=dtype)


def make_scaler():
    enabled = USE_AMP and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported()
    return torch.amp.GradScaler("cuda", enabled=enabled)


def lr_multiplier(epoch: int):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS - 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def validation_subject_macro_dice(model, loader):
    model.eval()
    records = []
    for batch in loader:
        ap = batch["ap"].to(DEVICE, non_blocking=True)
        lat = batch["lat"].to(DEVICE, non_blocking=True)
        target = batch["target"].to(DEVICE, non_blocking=True)
        with amp_context():
            logits = model(ap, lat)
        values = hard_dice(logits, target)[0].cpu().numpy()
        records.append({"subject_id": batch["subject_id"][0], "macro_dice": float(values.mean())})
    frame = pd.DataFrame(records)
    return float(frame.groupby("subject_id", as_index=False).macro_dice.mean().macro_dice.mean())


def save_curves(history, output_path, arm):
    frame = pd.DataFrame(history)
    fig, left = plt.subplots(figsize=(8, 4.5))
    right = left.twinx()
    left.plot(frame.epoch, frame.train_loss, color="tab:blue", label="Train loss")
    right.plot(frame.epoch, frame.validation_subject_macro_dice, color="tab:orange", label="Validation Dice")
    left.set(xlabel="Epoch", ylabel="Loss", title=f"{ARM_FACTORS[arm]['label']} training")
    right.set_ylabel("Subject-macro Dice")
    left.grid(alpha=0.25)
    lines = left.lines + right.lines
    left.legend(lines, [line.get_label() for line in lines], loc="best")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def train_arm(arm: str, simclr_checkpoint, pos_weight):
    output_dir = MODEL_ROOT / arm
    output_dir.mkdir(parents=True, exist_ok=True)
    best_path = output_dir / "best_model.pth"
    last_path = output_dir / "last_checkpoint.pth"
    history_path = output_dir / "history.csv"
    seed_everything()
    model = JointReconstructionModel(arm, simclr_checkpoint).to(DEVICE)
    encoder_parameters = list(model.encoder.parameters())
    encoder_ids = {id(parameter) for parameter in encoder_parameters}
    new_parameters = [parameter for parameter in model.parameters() if id(parameter) not in encoder_ids]
    optimizer = torch.optim.AdamW([
        {"params": encoder_parameters, "lr": ENCODER_LR},
        {"params": new_parameters, "lr": NEW_MODULE_LR},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
    scaler = make_scaler()

    train_set = ReconstructionDataset(splits["train"], training=True)
    validation_set = ReconstructionDataset(splits["validation"], training=False)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, generator=generator)
    validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)

    history, best_dice, stale, start_epoch = [], -math.inf, 0, 0
    if RESUME_TRAINING and last_path.is_file():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        if checkpoint["arm"] != arm:
            raise RuntimeError(f"Checkpoint arm mismatch: {checkpoint['arm']} != {arm}")
        model.load_state_dict(checkpoint["model_state"], strict=True)
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        scaler.load_state_dict(checkpoint["scaler_state"])
        history = checkpoint["history"]
        best_dice = float(checkpoint["best_dice"])
        stale = int(checkpoint["stale"])
        start_epoch = int(checkpoint["epoch"]) + 1
        print(f"Resuming {arm} at epoch {start_epoch}/{MAX_EPOCHS}")

    encoder_gradient_confirmed = False
    for epoch in range(start_epoch, MAX_EPOCHS):
        train_set.set_epoch(epoch)
        model.train()
        losses = []
        for batch in train_loader:
            ap = batch["ap"].to(DEVICE, non_blocking=True)
            lat = batch["lat"].to(DEVICE, non_blocking=True)
            target = batch["target"].to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = model(ap, lat)
                loss = dice_bce_loss(logits, target, pos_weight)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss for {arm} at epoch {epoch}.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if not encoder_gradient_confirmed:
                encoder_gradient_confirmed = any(
                    parameter.grad is not None and torch.isfinite(parameter.grad).all() and bool(parameter.grad.abs().sum() > 0)
                    for parameter in encoder_parameters
                )
                if not encoder_gradient_confirmed:
                    raise RuntimeError("The encoder did not receive a finite non-zero reconstruction gradient.")
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        validation_dice = validation_subject_macro_dice(model, validation_loader)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "validation_subject_macro_dice": validation_dice,
            "encoder_lr": optimizer.param_groups[0]["lr"],
            "new_module_lr": optimizer.param_groups[1]["lr"],
        }
        history.append(row)
        improved = validation_dice > best_dice
        if improved:
            best_dice, stale = validation_dice, 0
            torch.save({
                "arm": arm,
                "epoch": epoch,
                "validation_subject_macro_dice": validation_dice,
                "model_state": model.state_dict(),
                "model_name": simclr_checkpoint["model_name"],
                "normalization": simclr_checkpoint["normalization"],
            }, best_path)
        else:
            stale += 1
        scheduler.step()
        pd.DataFrame(history).to_csv(history_path, index=False)
        torch.save({
            "arm": arm,
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "history": history,
            "best_dice": best_dice,
            "stale": stale,
        }, last_path)
        save_curves(history, output_dir / "training_curves.png", arm)
        print(f"{arm} epoch={epoch + 1:02d}/{MAX_EPOCHS} loss={row['train_loss']:.5f} val_dice={validation_dice:.5f}")
        if stale >= PATIENCE:
            print(f"Early stopping {arm} after {PATIENCE} epochs without improvement.")
            break

    summary = {
        "arm": arm,
        "label": ARM_FACTORS[arm]["label"],
        "best_validation_subject_macro_dice": best_dice,
        "epochs_completed": len(history),
        "encoder_gradient_confirmed": encoder_gradient_confirmed,
        "best_model": str(best_path),
    }
    (output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    del model, optimizer, scheduler, scaler, train_loader, validation_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


def run_all_arms():
    if not SIMCLR_PATH.is_file():
        raise FileNotFoundError(f"Run notebook 00 first; checkpoint not found: {SIMCLR_PATH}")
    simclr_checkpoint = torch.load(SIMCLR_PATH, map_location="cpu", weights_only=False)
    pos_weight = training_pos_weight(splits["train"]).to(DEVICE)
    summaries = [train_arm(arm, simclr_checkpoint, pos_weight) for arm in ARMS_TO_RUN]
    frame = pd.DataFrame(summaries)
    frame.to_csv(MODEL_ROOT / "training_summary.csv", index=False)
    display(frame)
    return frame


if RUN_TRAINING:
    training_summary = run_all_arms()
else:
    print("Definitions loaded. Set RUN_TRAINING=True after notebook 00 has produced simclr_encoder.pth.")

plain_unet_style epoch=01/50 loss=0.95259 val_dice=0.17106
plain_unet_style epoch=02/50 loss=0.80772 val_dice=0.17335
plain_unet_style epoch=03/50 loss=0.73575 val_dice=0.18599
plain_unet_style epoch=04/50 loss=0.68400 val_dice=0.23489
plain_unet_style epoch=05/50 loss=0.64733 val_dice=0.23417
plain_unet_style epoch=06/50 loss=0.61310 val_dice=0.22001
plain_unet_style epoch=07/50 loss=0.59313 val_dice=0.27311
plain_unet_style epoch=08/50 loss=0.57109 val_dice=0.24947
plain_unet_style epoch=09/50 loss=0.55947 val_dice=0.28370
plain_unet_style epoch=10/50 loss=0.54154 val_dice=0.28231
plain_unet_style epoch=11/50 loss=0.53376 val_dice=0.28553
plain_unet_style epoch=12/50 loss=0.51619 val_dice=0.28625
plain_unet_style epoch=13/50 loss=0.50076 val_dice=0.30921
plain_unet_style epoch=14/50 loss=0.49473 val_dice=0.32573
plain_unet_style epoch=15/50 loss=0.48335 val_dice=0.33428
plain_unet_style epoch=16/50 loss=0.46598 val_dice=0.31590
plain_unet_style epoch=17/50 loss=0.45192 val_dice=0.326

,arm,label,best_validation_subject_macro_dice,epochs_completed,encoder_gradient_confirmed,best_model
0,plain_unet_style,"U (ReLU, plain)",0.411375,47,True,/home/project/xray2mesh/Marcus_Chan_Zheng_Shao...
1,plain_prelu_style,PReLU + plain,0.447960,49,True,/home/project/xray2mesh/Marcus_Chan_Zheng_Shao...
2,residual_relu_style,ReLU + residual,0.464338,43,True,/home/project/xray2mesh/Marcus_Chan_Zheng_Shao...
3,residual_vnet_style,"V (PReLU, residual)",0.483203,43,True,/home/project/xray2mesh/Marcus_Chan_Zheng_Shao...
